In [1]:
import ee
import geemap
ee.Initialize(project='wolfenden-maeve-biome-mapping')

In [2]:
def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)

In [3]:
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate("2018-01-01", "2022-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
    .map(mask_s2_clouds)
).select(["B2","B3","B4","B8"])

def add_ndvi(img):
    return img.addBands(
        img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    )

s2 = s2.map(add_ndvi)

composite = s2.median()
mosaic = s2.mosaic()

In [4]:
mosaic

In [5]:
def label_aoi(feature):
    biome_type = biomes.reduceRegion(
        reducer=ee.Reducer.mode(),
        geometry=feature.geometry(),
        scale=1000,
        maxPixels=1e13
    ).get("biome_type")

    return feature.set("biome_type", biome_type)

In [6]:
NUM_AOIS = 500
AOI_RADIUS_METERS = 500

biomes = ee.Image(
    "OpenLandMap/PNV/PNV_BIOME-TYPE_BIOME00K_C/v01"
)

# region=ee.Geometry.Rectangle([-180, -60, 0, 80])
# points = ee.FeatureCollection.randomPoints(
#     region=region,
#     points=NUM_AOIS,
#     seed=42)

# aois = points.map(lambda f: f.buffer(AOI_RADIUS_METERS).bounds())

# labeled_aois = (aois.map(label_aoi).filter(ee.Filter.notNull(["biome_type"])))

left_lats = [-180, -165, -150, -135, -120, -105, -90, -75, -60, -45 -30, -15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150]
right_lats = [-165, -150, -135, -120, -105, -90, -75, -60, -45 -30, -15, 0, 15, 30, 45, 60, 75, 90, 105, 120, 135, 150, 180]
labeled_aois_list = []

for i in range(len(left_lats)):
    print("Processing region", i,"/24")
    region=ee.Geometry.Rectangle([left_lats[i], -60, right_lats[i], 80])

    # m = geemap.Map()
    # m.add_layer(region, {'color': 'yellow'}, 'Region')
    points = ee.FeatureCollection.randomPoints(
    region=region,
    points=NUM_AOIS,
    seed=42)

    aois = points.map(
    lambda f: f.buffer(AOI_RADIUS_METERS).bounds())

    labeled_aois = (
    aois
    .map(label_aoi)
    .filter(ee.Filter.notNull(["biome_type"])))

    labeled_aois_list.append(labeled_aois)


print("Processing complete.")

Processing region 0 /24
Processing region 1 /24
Processing region 2 /24
Processing region 3 /24
Processing region 4 /24
Processing region 5 /24
Processing region 6 /24
Processing region 7 /24
Processing region 8 /24
Processing region 9 /24
Processing region 10 /24
Processing region 11 /24
Processing region 12 /24
Processing region 13 /24
Processing region 14 /24
Processing region 15 /24
Processing region 16 /24
Processing region 17 /24
Processing region 18 /24
Processing region 19 /24
Processing region 20 /24
Processing region 21 /24
Processing complete.


In [7]:
labeled_aois = labeled_aois_list[0]
for i in range(0, len(labeled_aois_list)-1):
    labeled_aois = labeled_aois_list[i].merge(labeled_aois_list[i+1])

In [8]:
labeled_aois

In [9]:
# m = geemap.Map()
# m.add_layer(region, {'color': 'yellow'}, 'Region')
# m.add_layer(points, {'color': 'black'}, 'Random points')
# m

In [10]:
# Rasterize AOI biome labels
biome_label_image = ee.Image().byte().paint(
    featureCollection=labeled_aois,
    color="biome"
).rename("biome")


In [11]:
PATCH_SIZE = 32
SCALE = 10

input_bands = ["B2", "B3", "B4", "B8", "NDVI"]

stacked_image = (
    composite.select(input_bands)
    .addBands(biome_label_image)
)


In [12]:
stacked_image

In [13]:

# visualization_1 = {
#     'min': 0.0,
#     'max': 0.3,
#     'bands': ['B4'],
# }

# visualization_2 = {
#   "bands": ['biome'],
#   min: 1.0,
#   max: 32.0,
#   "palette": [
#     '1c5510','659208','ae7d20','000065','bbcb35','009a18',
#     'caffca','55eb49','65b2ff','0020ca','8ea228','ff9adf',
#     'baff35','ffba9a','ffba35','f7ffca','e7e718','798649',
#     '65ff9a','d29e96',
#   ]
# };

# m = geemap.Map()
# m.set_center(138.74362721224972, 57.52490322923566, 18)
# #m.add_layer(stacked_image, visualization_1, 'R')
# m.add_layer(stacked_image, visualization_1, 'R')
# m.addLayer(stacked_image, visualization_2)

# m

In [14]:
labeled_aois

In [17]:
sample = composite.sampleRegions(
  collection=labeled_aois,
  properties=["biome_type"],
  scale=50).randomColumn()

In [18]:
training = sample.filter(ee.Filter.lt('random', 0.7))
testing = sample.filter(ee.Filter.gte('random', 0.7))

# Print the first couple points to verify.
from pprint import pprint
pprint({'training': training.first().getInfo()})
pprint({'testing': testing.first().getInfo()})

{'training': {'geometry': None,
              'id': '1_5_0',
              'properties': {'B2': 0.05567999929189682,
                             'B3': 0.07355000078678131,
                             'B4': 0.09860000014305115,
                             'B8': 0.2012999951839447,
                             'NDVI': 0.3348641097545624,
                             'biome_type': 16,
                             'random': 0.3760402383210416},
              'type': 'Feature'}}
{'testing': {'geometry': None,
             'id': '1_5_5',
             'properties': {'B2': 0.07680000364780426,
                            'B3': 0.10378000140190125,
                            'B4': 0.12995000183582306,
                            'B8': 0.29019999504089355,
                            'NDVI': 0.28958022594451904,
                            'biome_type': 16,
                            'random': 0.9059959499051417},
             'type': 'Feature'}}


In [17]:
# Map = geemap.Map()
# Map.addLayer(stacked_image.select("B4"), {"min": 0, "max": 3000}, "Red")
# Map.addLayer(biome_label_image, {"min": 1, "max": 8}, "Biome Labels")
# m.set_center(120.74333495614178, 27.868199457432233)
# Map

In [18]:
task = ee.batch.Export.table.toDrive(
    collection=training,
    description='Training Export',
    folder="cnn_biomes",
    fileNamePrefix="training",
    #region=labeled_aois.geometry(),
    fileFormat="TFRecord",
)

task.start()

task = ee.batch.Export.table.toDrive(
    collection=testing,
    description='Testing Export',
    folder="cnn_biomes",
    fileNamePrefix="testing",
    #region=labeled_aois.geometry(),
    fileFormat="TFRecord",
)

task.start()


In [19]:
task.status()

{'state': 'COMPLETED',
 'description': 'Testing Export',
 'priority': 100,
 'creation_timestamp_ms': 1767081831879,
 'update_timestamp_ms': 1767081857343,
 'start_timestamp_ms': 1767081847719,
 'task_type': 'EXPORT_FEATURES',
 'destination_uris': ['https://drive.google.com/#folders/1VrjGW-Nby37OkrMZoIM_gR0tCLO9VD4E'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 6.946743488311768,
 'id': 'KATMANFW5OONRU3H7BTTGF3O',
 'name': 'projects/wolfenden-maeve-biome-mapping/operations/KATMANFW5OONRU3H7BTTGF3O'}